# Deflection ΔV Calculator Example Notebook

This notebook calls functions provided in **deflection_propogation.py** to calculate the minimum ΔV (and corresponding time) required to deflect a heliocentric body to some distance away from a main heliocentric body on the Opik B-Plane.

Both **kinetic** and **continuous** impulse deflection examples are shown below.

In the example, "main" corresponds to Earth and "secondary" corresponds to the asteroid.

#### Imports

In [1]:
import numpy as np
from astropy.time import Time
from deflection_propogation import (
    au2km,
    km2au,
    au_per_day2km_per_s,
    get_nominal_sof_and_bplane_grss,
    search_best_continuous_start_time_grss,
    search_best_impulse_time_grss,
    plot_bplane,
    init_grss_sim,
)

/home/wschafer1/projects/AE498PD_FinalProject/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Orbital Elements & Deflection Parameters

In [2]:
t0_mjd = 60522.0  # MJD TDB --- 2024-07-31 --- impact is confirmed to be 2041-04-24

secondary_sol = {
    "t": t0_mjd,
    "e": 0.390657994905,
    "q": 1.00538004981,
    "tp": 60440.5982662,  # MJD TDB
    "om": np.deg2rad(214.423769139),
    "w": np.deg2rad(359.963802784),
    "i": np.deg2rad(10.6888293122),
}

secondary_cov = np.array([
    [ 9.16537871e-13, -1.63466743e-12,  3.89020101e-11,  2.48737127e-12, -2.63252366e-12, -5.36397528e-13],
    [-1.63466743e-12,  3.13128877e-12,  6.19207457e-12, -4.57191920e-12, 6.49897965e-12,  8.38289521e-13],
    [ 3.89020101e-11,  6.19207457e-12,  7.01455580e-08, -3.17541559e-10, 1.48376403e-09, -1.27526089e-10],
    [ 2.48737127e-12, -4.57191920e-12, -3.17541559e-10,  2.28913017e-10, -2.27363461e-10,  2.17832920e-11],
    [-2.63252366e-12,  6.49897965e-12,  1.48376403e-09, -2.27363461e-10, 2.49311599e-10, -2.35745507e-11],
    [-5.36397528e-13,  8.38289521e-13, -1.27526089e-10,  2.17832920e-11, -2.35745507e-11,  5.55074093e-12]
], dtype=float)

mu_main = 8.8877e-10               # AU^3 / day^2
R_main = 0.0000426354              # AU
R_main_sof = 0.006216666023709612  # AU

t_max = 30.0 * 365.0
dt_search_imp = 1.0
dt_search_kin = 1.0
b_main = 4.0 * R_main

#### Time Entering Sphere of Influence (for time search optimization)

In [3]:
nominal = get_nominal_sof_and_bplane_grss(sol_secondary=secondary_sol, mu_main=mu_main, R_main=R_main, R_sof=R_main_sof, t_max=t_max, dt=dt_search_imp)
if nominal is None:
    raise RuntimeError("No nominal SOF crossings found with GRSS propagation.")

print("")
print(f"Found {len(nominal['encounters'])} SOF encounters:\n")
for i, enc in enumerate(nominal["encounters"]):
    t_entry_date = Time(enc["t_entry_abs"], format="mjd", scale="tdb").utc.iso[:10]
    t_ca_date = Time(enc["t_ca_abs"], format="mjd", scale="tdb").utc.iso[:10]
    t_exit_date = Time(enc["t_exit_abs"], format="mjd", scale="tdb").utc.iso[:10]
    print(f"===== ENCOUNTER {i+1} =====")
    print(f"Relative SOF Entry Time: {enc['t_entry_rel']:.6f} d")
    print(f"Absolute SOF Entry Time: {enc['t_entry_abs']:.6f} MJD")
    print(f"SOF Entry Date: {t_entry_date}")
    print(f"Relative Closest Approach Time: {enc['t_ca_rel']:.6f} d")
    print(f"Absolute Closest Approach Time: {enc['t_ca_abs']:.6f} MJD")
    print(f"Closest Approach Date: {t_ca_date}")
    print(f"Relative SOF Exit Time: {enc['t_exit_rel']:.6f} d")
    print(f"Absolute SOF Exit Time: {enc['t_exit_abs']:.6f} MJD")
    print(f"SOF Exit Date: {t_exit_date}")
    print(f"Closest Approach Distance: {enc['d_ca']:.6e} AU")
    print(f"B-Plane Magnitude: {enc['b']:.6e} AU")
    print("")

t_sof_target = nominal["encounters"][0]["t_entry_rel"]


Found 1 SOF encounters:

===== ENCOUNTER 1 =====
Relative SOF Entry Time: 6110.397072 d
Absolute SOF Entry Time: 66632.397072 MJD
SOF Entry Date: 2041-04-23
Relative Closest Approach Time: 6111.688140 d
Absolute Closest Approach Time: 66633.688140 MJD
Closest Approach Date: 2041-04-24
Relative SOF Exit Time: 6112.977412 d
Absolute SOF Exit Time: 66634.977412 MJD
SOF Exit Date: 2041-04-25
Closest Approach Distance: 7.867622e-06 AU
B-Plane Magnitude: 2.299608e-05 AU



---
## Continuous Deflection Example

In [ ]:
best_continuous_deflections = {}
for direction in ["radial", "transverse", "normal"]:
    best_continuous_deflections[direction] = search_best_continuous_start_time_grss(
        sol_secondary=secondary_sol,          # asteroid GRSS solution dict
        cov_secondary=secondary_cov,          # asteroid covariance in GRSS order [e, q, tp, om, w, i]
        mu_main=mu_main,                      # Earth gravitational parameter [AU^3 / day^2]
        R_main=R_main,                        # Earth radius [AU]
        R_sof=R_main_sof,                     # sphere-of-influence radius used for encounter definition [AU]
        direction=direction,                  # applied thrust direction: radial / transverse / normal
        b_target=b_main,                      # desired minimum b-plane miss distance [AU]
        t_start_min=0.0,                      # earliest continuous-thrust start time [days after epoch]
        t_start_max=t_sof_target,             # latest start time allowed: chosen SOF entry [days after epoch]
        t_max=t_max,                          # propagation horizon [days after epoch]
        dt=dt_search_kin,                     # time step used inside SOF/crossing searches [days]
        burn_step=2.0,                       # piecewise-burn segment size [days]
        coarse_step=60.0,                     # coarse search spacing for start-time sweep [days]
        mid_half_width=10.0,                   # half-width of mid-level refinement window [days]
        mid_step=10.0,                         # mid-level search spacing [days]
        fine_half_width=2.0,                  # half-width of fine refinement window [days]
        fine_step=2.0,                        # fine-level search spacing [days]
        robust_metric="min_b",                # optimize against worst-case sigma-point b-plane miss distance
    )

print("")
print("Best Continuous Deflections:\n")
for direction, sol in best_continuous_deflections.items():
    sign_label = "+" if sol["deltaV"] >= 0 else "-"
    print(f"{direction.capitalize()} Continuous")
    print(f"Sign: {sign_label}")
    print(f"Relative Thrust Start Time: {sol['t_start']:.6f} d")
    print(f"Minimum Required Acceleration-Integrated DeltaV: {au_per_day2km_per_s(abs(sol['deltaV'])):.6e} km/s")
    print("")

NameError: name 't_sof_nominal' is not defined

#### B-Plane Plot

In [ ]:
for direction, sol in best_continuous_deflections.items():
    case = sol["b_elems"]
    xi = case["xi"]
    zeta = case["zeta"]
    b_coll = case["b_coll"]
    plot_bplane(au2km(xi), au2km(zeta), au2km(b_coll), au2km(R_main), title=f"Optimized Continuous B-plane ({direction})", units_in_km=True)

---
## Kinetic Deflection Example

In [ ]:
best_impulse_deflections = {}
for direction in ["radial", "transverse", "normal"]:
    best_impulse_deflections[direction] = search_best_impulse_time_grss(
        sol_secondary=secondary_sol,          # asteroid GRSS solution dict
        cov_secondary=secondary_cov,          # asteroid covariance in GRSS order [e, q, tp, om, w, i]
        mu_main=mu_main,                      # Earth gravitational parameter [AU^3 / day^2]
        R_main=R_main,                        # Earth radius [AU]
        R_sof=R_main_sof,                     # sphere-of-influence radius used for encounter definition [AU]
        direction=direction,                  # applied impulse direction: radial / transverse / normal
        b_target=b_main,                      # desired minimum b-plane miss distance [AU]
        t_impulse_min=0.0,                    # earliest impulse time [days after epoch]
        t_impulse_max=t_sof_target,           # latest impulse time allowed: chosen SOF entry [days after epoch]
        t_max=t_max,                          # propagation horizon [days after epoch]
        dt=dt_search_imp,                     # time step used inside SOF/crossing searches [days]
        coarse_step=60.0,                     # coarse search spacing for impulse-time sweep [days]
        mid_half_width=10.0,                   # half-width of mid-level refinement window [days]
        mid_step=10.0,                         # mid-level search spacing [days]
        fine_half_width=2.0,                  # half-width of fine refinement window [days]
        fine_step=2.0,                        # fine-level search spacing [days]
        robust_metric="min_b",                # optimize against worst-case sigma-point b-plane miss distance
    )

print("")
print("Best Impulse Deflections:\n")
for direction, sol in best_impulse_deflections.items():
    sign_label = "+" if sol["dV_min"] >= 0 else "-"
    print(f"{direction.capitalize()} Impulse")
    print(f"Sign: {sign_label}")
    print(f"Relative Impulse Time: {sol['t_impulse']:.6f} d")
    print(f"Minimum Required Impulse DeltaV: {au_per_day2km_per_s(abs(sol['dV_min'])):.6e} km/s")
    print("")

#### B-Plane Plot

In [ ]:
for direction, sol in best_impulse_deflections.items():
    case = sol["b_elems"]
    xi = case["xi"]
    zeta = case["zeta"]
    b_coll = case["b_coll"]
    plot_bplane(au2km(xi), au2km(zeta), au2km(b_coll), au2km(R_main), title=f"Optimized Impulse B-plane ({direction})", units_in_km=True)